In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

print("Libraries loaded.")

print(
    "\nNotebook mode: reporting/visualization only. "
    "No retraining, tuning, or final-test rerun."
)

In [ ]:
# ============================================================
# 2. FIND AFTERLIFE-AI REPOSITORY ROOT — PORTABLE LOCAL
# ============================================================

import os
import subprocess

# Optional escape hatch:
# If an IDE launches Jupyter from a strange working directory,
# set this to the cloned repository root. Normally leave as None.
MANUAL_REPO_ROOT: str | None = None


def looks_like_afterlife_repo(path: Path) -> bool:
    """Return True only for the expected Afterlife-AI repo structure."""
    return (
        (path / "pyproject.toml").exists()
        and (path / "data/generated/synthetic_candidates_v2.csv").exists()
        and (path / "reports/evidence/modeling").exists()
        and (path / "src/afterlife_ai").exists()
    )


def git_repo_root(start: Path) -> Path | None:
    """Use Git itself when the kernel is already somewhere inside the repo."""
    try:
        result = subprocess.run(
            ["git", "-C", str(start), "rev-parse", "--show-toplevel"],
            capture_output=True,
            text=True,
            check=True,
        )
    except (OSError, subprocess.CalledProcessError):
        return None

    candidate = Path(result.stdout.strip()).resolve()

    if looks_like_afterlife_repo(candidate):
        return candidate

    return None


def find_repo_root() -> Path:
    # 1. Explicit override for unusual IDE/Jupyter setups.
    if MANUAL_REPO_ROOT:
        candidate = Path(MANUAL_REPO_ROOT).expanduser().resolve()

        if not looks_like_afterlife_repo(candidate):
            raise RuntimeError(
                "MANUAL_REPO_ROOT sudah diisi tetapi bukan root repo Afterlife-AI:\n"
                f"{candidate}"
            )

        return candidate

    # 2. Optional environment variable for CI / another machine.
    env_root = os.environ.get("AFTERLIFE_AI_REPO")

    if env_root:
        candidate = Path(env_root).expanduser().resolve()

        if looks_like_afterlife_repo(candidate):
            return candidate

    current = Path.cwd().resolve()

    # 3. Most portable case: notebook/kernel is inside cloned Git repo.
    candidate = git_repo_root(current)

    if candidate is not None:
        return candidate

    # 4. Check current folder and its parents directly.
    for base in [current, *current.parents]:
        if looks_like_afterlife_repo(base):
            return base

    # 5. Common workspace layouts.
    # This intentionally searches only deterministic relative locations,
    # not the entire disk.
    relative_patterns = [
        Path("repo"),
        Path("Afterlife-AI"),
        Path("07_PRODUCTION") / "repo",
        Path("Afterlife-AI") / "07_PRODUCTION" / "repo",
    ]

    for base in [current, *list(current.parents)[:5]]:
        for relative in relative_patterns:
            candidate = (base / relative).resolve()

            if looks_like_afterlife_repo(candidate):
                return candidate

    raise RuntimeError(
        "Afterlife-AI repository root tidak ditemukan.\n\n"
        f"Current working directory: {current}\n\n"
        "Cara paling portable:\n"
        "1. simpan notebook ini di <repo>/notebooks/ lalu buka folder repo sebagai project; atau\n"
        "2. set MANUAL_REPO_ROOT di cell ini ke root clone lokal; atau\n"
        "3. set environment variable AFTERLIFE_AI_REPO.\n\n"
        "Tidak ada path laptop tertentu yang di-hardcode."
    )


ROOT = find_repo_root()

print("Current working directory:")
print(Path.cwd())

print("\nRepository root:")
print(ROOT)

print("\nRepository structure: OK")

In [ ]:
# ============================================================
# 3. DEFINE EVIDENCE PATHS
# ============================================================

PATHS = {
    "candidates": ROOT / "data/generated/synthetic_candidates_v2.csv",
    "oracle": ROOT / "data/generated/synthetic_oracle_v2.csv",
    "split_manifest": ROOT / "reports/evidence/synthetic_dataset/SPLIT_GROUPS_v2.csv",
    "baseline_metrics": ROOT / "reports/evidence/modeling/BASELINE_VALIDATION_METRICS_v1.json",
    "baseline_predictions": ROOT / "reports/evidence/modeling/BASELINE_VALIDATION_PREDICTIONS_v1.csv",
    "hgb_ablation": ROOT / "reports/evidence/modeling/HGB_ABLATION_VALIDATION_v1.json",
    "selection_scores": ROOT / "reports/evidence/modeling/VALIDATION_MODEL_SELECTION_SCORES_v1.csv",
    "model_allocation": ROOT / "reports/evidence/modeling/VALIDATION_ALLOCATION_REGRET_v1.json",
    "baseline_allocation": ROOT / "reports/evidence/modeling/BASELINE_VALIDATION_ALLOCATION_REGRET_v1.json",
    "robustness": ROOT / "reports/evidence/modeling/AI_VALUE_GATE_ROBUSTNESS_v1.json",
    "selected_manifest": ROOT / "reports/evidence/modeling/SELECTED_MODEL_MANIFEST_v1.json",
    "final_test": ROOT / "reports/evidence/modeling/final_test/FINAL_LOCKED_TEST_v1.json",
    "final_predictions": ROOT / "reports/evidence/modeling/final_test/FINAL_LOCKED_TEST_PREDICTIONS_v1.csv",
    "final_allocation": ROOT / "reports/evidence/modeling/final_test/FINAL_LOCKED_TEST_ALLOCATION_REGRET_v1.csv",
}

missing = [name for name, path in PATHS.items() if not path.exists()]

if missing:
    print("Missing evidence:")
    for name in missing:
        print("-", name, "->", PATHS[name])
else:
    print("All required evidence files found.")
print("Evidence source:", ROOT)

# ============================================================
# 3B. NOTEBOOK OUTPUT DIRECTORIES
# ============================================================

REPORT_DIR = (
    ROOT
    / "reports"
    / "figures"
    / "notebook_visualizations"
)

TEXT_DIR = (
    ROOT
    / "reports"
    / "evidence"
    / "notebook_text_evidence"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TEXT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def save_current_figure(filename: str) -> Path:
    """Save the active Matplotlib figure as reproducible notebook evidence."""
    path = REPORT_DIR / filename

    plt.savefig(
        path,
        dpi=200,
        bbox_inches="tight",
    )

    print(
        "Saved figure:",
        path.relative_to(ROOT),
    )

    return path


def save_text_evidence(
    filename: str,
    content: str,
) -> Path:
    """Save notebook-style textual evidence as UTF-8."""
    path = TEXT_DIR / filename

    if not content.endswith("\n"):
        content += "\n"

    path.write_text(
        content,
        encoding="utf-8",
    )

    print(
        "Saved text evidence:",
        path.relative_to(ROOT),
    )

    return path


print("\nNotebook visualization directory:")
print(REPORT_DIR)

print("\nNotebook text evidence directory:")
print(TEXT_DIR)


In [ ]:
# ============================================================
# 4. LOAD DATASET AND SPLIT MANIFEST
# ============================================================

candidates = pd.read_csv(PATHS["candidates"])
split_manifest = pd.read_csv(PATHS["split_manifest"])

dataset = candidates.merge(
    split_manifest[["scenario_group_id", "split"]],
    on="scenario_group_id",
    how="left",
    validate="many_to_one",
)

print("Dataset shape:", dataset.shape)
print("Scenario groups:", dataset["scenario_group_id"].nunique())
print("Candidate rows:", len(dataset))

display(dataset.head())

In [ ]:
# ============================================================
# 5. BASIC DATASET SANITY CHECK
# ============================================================

print("Missing values:")
display(dataset.isnull().sum())

print("\nDuplicate rows:")
print(dataset.duplicated().sum())

print("\nUnique scenario groups:")
print(dataset["scenario_group_id"].nunique())

print("\nUnique candidate IDs:")
print(dataset["candidate_id"].nunique())

print("\nLabel distribution:")
display(dataset["simulated_rescue_outcome"].value_counts())

print("\nLabel distribution percentage:")
display(
    (
        dataset["simulated_rescue_outcome"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )
)

dataset_sanity_text = "\n".join(
    [
        f"Dataset shape: {dataset.shape}",
        f"Scenario groups: {dataset['scenario_group_id'].nunique()}",
        f"Candidate rows: {len(dataset)}",
        "",
        "Missing values:",
        dataset.isnull().sum().to_string(),
        "",
        f"Duplicate rows: {int(dataset.duplicated().sum())}",
        f"Unique candidate IDs: {dataset['candidate_id'].nunique()}",
        "",
        "Label distribution:",
        dataset["simulated_rescue_outcome"].value_counts().to_string(),
        "",
        "Label distribution percentage:",
        (
            dataset["simulated_rescue_outcome"]
            .value_counts(normalize=True)
            .mul(100)
            .round(2)
            .to_string()
        ),
    ]
)

save_text_evidence(
    "01_DATASET_SANITY.txt",
    dataset_sanity_text,
)

In [ ]:
# ============================================================
# 6. GROUPED SPLIT SUMMARY
# ============================================================

split_summary = (
    dataset.groupby("split")
    .agg(
        scenario_groups=("scenario_group_id", "nunique"),
        candidate_rows=("candidate_id", "size"),
        positive_rate=("simulated_rescue_outcome", "mean"),
    )
    .reindex(["train", "validation", "test"])
)

display(split_summary)

save_text_evidence(
    "02_SPLIT_SUMMARY.txt",
    split_summary.to_string(),
)

In [ ]:
# ============================================================
# 7. SCENARIO GROUP COUNT BY SPLIT
# ============================================================

plt.figure(figsize=(8, 5))
plt.bar(
    split_summary.index,
    split_summary["scenario_groups"],
)
plt.title("Scenario Group Count by Split")
plt.xlabel("Split")
plt.ylabel("Scenario Groups")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("01_scenario_groups_by_split.png")
plt.show()

In [ ]:
# ============================================================
# 8. CANDIDATE ROW COUNT BY SPLIT
# ============================================================

plt.figure(figsize=(8, 5))
plt.bar(
    split_summary.index,
    split_summary["candidate_rows"],
)
plt.title("Candidate Row Count by Split")
plt.xlabel("Split")
plt.ylabel("Candidate Rows")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("02_candidate_rows_by_split.png")
plt.show()

In [ ]:
# ============================================================
# 9. POSITIVE LABEL RATE BY SPLIT
# ============================================================

plt.figure(figsize=(8, 5))
plt.bar(
    split_summary.index,
    split_summary["positive_rate"],
)
plt.title("Synthetic Rescue Outcome Rate by Split")
plt.xlabel("Split")
plt.ylabel("Positive Rate")
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("03_positive_rate_by_split.png")
plt.show()

In [ ]:
# ============================================================
# 10. CANDIDATE COUNT BY ACTION TYPE
# ============================================================

action_counts = (
    dataset["action_type"]
    .value_counts()
    .sort_values(ascending=False)
)

display(action_counts)

plt.figure(figsize=(11, 5))
action_counts.plot(kind="bar")
plt.title("Synthetic Candidate Count by Action Type")
plt.xlabel("Action Type")
plt.ylabel("Candidate Rows")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("04_action_type_distribution.png")
plt.show()

In [ ]:
# ============================================================
# 11. SYNTHETIC OUTCOME RATE BY ACTION TYPE
# ============================================================

action_outcome_rate = (
    dataset.groupby("action_type")["simulated_rescue_outcome"]
    .mean()
    .sort_values()
)

display(action_outcome_rate)

plt.figure(figsize=(10, 6))
action_outcome_rate.plot(kind="barh")
plt.title("Synthetic Rescue Outcome Rate by Action Type")
plt.xlabel("Positive Rate")
plt.ylabel("Action Type")
plt.xlim(0, 1)
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
save_current_figure("05_outcome_rate_by_action.png")
plt.show()

action_text = "\n".join(
    [
        "CANDIDATE COUNT BY ACTION TYPE",
        "=" * 70,
        action_counts.to_string(),
        "",
        "SYNTHETIC OUTCOME RATE BY ACTION TYPE",
        "=" * 70,
        action_outcome_rate.to_string(),
    ]
)

save_text_evidence(
    "03_ACTION_DISTRIBUTION.txt",
    action_text,
)

In [ ]:
# ============================================================
# 12. CANDIDATES PER SCENARIO GROUP
# ============================================================

candidate_count_per_group = (
    dataset.groupby("scenario_group_id")
    .size()
)

display(candidate_count_per_group.describe())

plt.figure(figsize=(8, 5))
plt.hist(candidate_count_per_group, bins=7)
plt.title("Candidate Count per Scenario Group")
plt.xlabel("Candidates in Group")
plt.ylabel("Scenario Groups")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("06_candidates_per_group.png")
plt.show()

In [ ]:
# ============================================================
# 13. NUMERIC FEATURE DISTRIBUTION SUMMARY
# ============================================================

feature_columns = [
    "planning_quantity",
    "remaining_shelf_life_days",
    "remaining_safe_window_hours",
    "remaining_commercial_window_days",
    "unit_cost",
    "normal_selling_price",
    "offered_or_selling_price_per_unit",
    "direct_action_cost",
    "logistics_cost",
    "handling_cost",
    "estimated_completion_hours",
    "active_demand_quantity",
    "available_capacity",
    "minimum_order_quantity",
    "capability_resource_ratio",
    "demand_coverage_ratio",
    "demand_freshness_hours",
    "distance_km",
]

available_features = [
    col for col in feature_columns
    if col in dataset.columns
]

display(
    dataset[available_features]
    .describe()
    .T
)

numeric_feature_summary = (
    dataset[available_features]
    .describe()
    .T
)

save_text_evidence(
    "04_NUMERIC_FEATURE_SUMMARY.txt",
    numeric_feature_summary.to_string(),
)

In [ ]:
# ============================================================
# 14. REMAINING SHELF LIFE DISTRIBUTION
# ============================================================

plt.figure(figsize=(8, 5))
plt.hist(
    dataset["remaining_shelf_life_days"],
    bins=20,
)
plt.title("Remaining Shelf Life Distribution")
plt.xlabel("Remaining Shelf Life (days)")
plt.ylabel("Candidate Rows")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("07_shelf_life_distribution.png")
plt.show()

In [ ]:
# ============================================================
# 15. DISTANCE DISTRIBUTION
# ============================================================

plt.figure(figsize=(8, 5))
plt.hist(
    dataset["distance_km"],
    bins=20,
)
plt.title("Candidate Distance Distribution")
plt.xlabel("Distance (km)")
plt.ylabel("Candidate Rows")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("08_distance_distribution.png")
plt.show()

In [ ]:
# ============================================================
# 16. BASELINE VALIDATION METRICS
# ============================================================

with open(PATHS["baseline_metrics"], "r", encoding="utf-8") as handle:
    baseline_metrics = json.load(handle)

baseline_rows = []

for baseline_id, metrics in baseline_metrics["baselines"].items():
    baseline_rows.append(
        {
            "baseline": baseline_id,
            "MRR": metrics["mrr"],
            "NDCG@3": metrics["ndcg_at_3"],
            "Top-1": metrics["top1_success_rate"],
        }
    )

baseline_ranking_df = (
    pd.DataFrame(baseline_rows)
    .set_index("baseline")
)

display(baseline_ranking_df)

save_text_evidence(
    "05_BASELINE_METRICS.txt",
    baseline_ranking_df.to_string(),
)

In [ ]:
# ============================================================
# 17. BASELINE RANKING COMPARISON
# ============================================================

baseline_ranking_df.plot(
    kind="bar",
    figsize=(9, 5),
)

plt.title("Validation Baseline Ranking Comparison")
plt.xlabel("Baseline")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("09_baseline_ranking.png")
plt.show()

In [ ]:
# ============================================================
# 18. FINAL MODEL SELECTION TABLE
# ============================================================

selection_df = pd.DataFrame(
    {
        "Model": ["Logistic Regression", "HGB-B", "HGB-E"],
        "PR-AUC": [0.849653, 0.858194, 0.853664],
        "Brier": [0.156055, 0.155372, 0.155145],
        "Top-1": [0.869444, 0.861111, 0.847222],
    }
)

display(selection_df)

model_selection_text = "\n".join(
    [
        selection_df.to_string(index=False),
        "",
        "Selected model: HGB-E",
        (
            "Selection rule: best validation Brier among models "
            "within 1% relative PR-AUC of the best model."
        ),
    ]
)

save_text_evidence(
    "06_MODEL_SELECTION.txt",
    model_selection_text,
)

In [ ]:
# ============================================================
# 19. VALIDATION PR-AUC MODEL COMPARISON
# ============================================================

plt.figure(figsize=(9, 5))
plt.bar(
    selection_df["Model"],
    selection_df["PR-AUC"],
)
plt.title("Validation Model Comparison Based on PR-AUC")
plt.xlabel("Model")
plt.ylabel("PR-AUC")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("10_validation_pr_auc.png")
plt.show()

In [ ]:
# ============================================================
# 20. VALIDATION BRIER MODEL COMPARISON
# ============================================================

plt.figure(figsize=(9, 5))
plt.bar(
    selection_df["Model"],
    selection_df["Brier"],
)
plt.title("Validation Model Comparison Based on Brier Score")
plt.xlabel("Model")
plt.ylabel("Brier Score (Lower is Better)")
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("11_validation_brier.png")
plt.show()

In [ ]:
# ============================================================
# 21. PR-AUC VS BRIER MODEL SELECTION VIEW
# ============================================================

plt.figure(figsize=(8, 6))
plt.scatter(
    selection_df["PR-AUC"],
    selection_df["Brier"],
    s=100,
)

for _, row in selection_df.iterrows():
    plt.annotate(
        row["Model"],
        (row["PR-AUC"], row["Brier"]),
        xytext=(6, 6),
        textcoords="offset points",
    )

plt.title("Validation Model Selection: PR-AUC vs Brier")
plt.xlabel("PR-AUC (Higher is Better)")
plt.ylabel("Brier Score (Lower is Better)")
plt.grid(alpha=0.25)
plt.tight_layout()
save_current_figure("12_pr_auc_vs_brier.png")
plt.show()

print("Selected model: HGB-E")

In [ ]:
# ============================================================
# 22. LOAD AI VALUE GATE ROBUSTNESS EVIDENCE
# ============================================================

with open(PATHS["robustness"], "r", encoding="utf-8") as handle:
    robustness = json.load(handle)

robustness_df = pd.DataFrame(
    robustness["seed_results"]
)

display(
    robustness_df[
        [
            "robustness_seed",
            "hgb_e_pr_auc",
            "b1_pr_auc",
            "delta_pr_auc",
            "bootstrap_ci_95_low",
            "bootstrap_ci_95_high",
            "hgb_e_brier",
            "b1_brier",
            "seed_consistent",
        ]
    ]
)

In [ ]:
# ============================================================
# 23. ROBUSTNESS PR-AUC COMPARISON BY SEED
# ============================================================

robustness_compare = (
    robustness_df[
        [
            "robustness_seed",
            "hgb_e_pr_auc",
            "b1_pr_auc",
        ]
    ]
    .set_index("robustness_seed")
    .rename(
        columns={
            "hgb_e_pr_auc": "HGB-E",
            "b1_pr_auc": "B1",
        }
    )
)

robustness_compare.plot(
    kind="bar",
    figsize=(9, 5),
)

plt.title("Robustness PR-AUC: HGB-E vs B1")
plt.xlabel("Robustness Seed")
plt.ylabel("PR-AUC")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("13_robustness_pr_auc.png")
plt.show()

In [ ]:
# ============================================================
# 24. ROBUSTNESS PR-AUC DELTA WITH 95% CI
# ============================================================

x = np.arange(len(robustness_df))

delta = robustness_df["delta_pr_auc"].to_numpy()
lower = robustness_df["bootstrap_ci_95_low"].to_numpy()
upper = robustness_df["bootstrap_ci_95_high"].to_numpy()

yerr = np.vstack(
    [
        delta - lower,
        upper - delta,
    ]
)

plt.figure(figsize=(8, 5))
plt.errorbar(
    x,
    delta,
    yerr=yerr,
    fmt="o",
    capsize=5,
)
plt.axhline(0, linewidth=1)

plt.xticks(
    x,
    robustness_df["robustness_seed"].astype(str),
)

plt.title("HGB-E PR-AUC Improvement vs B1")
plt.xlabel("Robustness Seed")
plt.ylabel("Delta PR-AUC")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("14_robustness_ci.png")
plt.show()

In [ ]:
# ============================================================
# 25. AI VALUE GATE SUMMARY
# ============================================================

aggregate = robustness["aggregate_metrics"]
gate = robustness["gate"]

print("HGB-E mean PR-AUC:", round(aggregate["hgb_e_pr_auc_mean"], 6))
print("B1 mean PR-AUC:", round(aggregate["b1_pr_auc_mean"], 6))
print("Mean PR-AUC improvement:", round(aggregate["mean_delta_pr_auc"], 6))

print("\nHGB-E mean Brier:", round(aggregate["hgb_e_brier_mean"], 6))
print("B1 mean Brier:", round(aggregate["b1_brier_mean"], 6))
print("Mean Brier difference:", round(aggregate["mean_delta_brier"], 6))

print("\nAggregate PR-AUC delta 95% CI:")
print(
    round(aggregate["aggregate_delta_pr_auc_ci_95"]["low"], 6),
    "to",
    round(aggregate["aggregate_delta_pr_auc_ci_95"]["high"], 6),
)

print("\nConsistent seeds:", aggregate["consistent_seed_count"], "/ 3")
print("AI VALUE GATE:", "PASS" if gate["passed"] else "NOT PASSED")

ai_value_gate_text = "\n".join(
    [
        f"HGB-E mean PR-AUC: {aggregate['hgb_e_pr_auc_mean']:.6f}",
        f"B1 mean PR-AUC: {aggregate['b1_pr_auc_mean']:.6f}",
        f"Mean PR-AUC improvement: {aggregate['mean_delta_pr_auc']:.6f}",
        "",
        f"HGB-E mean Brier: {aggregate['hgb_e_brier_mean']:.6f}",
        f"B1 mean Brier: {aggregate['b1_brier_mean']:.6f}",
        f"Mean Brier difference: {aggregate['mean_delta_brier']:.6f}",
        "",
        (
            "Aggregate PR-AUC delta 95% CI: "
            f"{aggregate['aggregate_delta_pr_auc_ci_95']['low']:.6f} "
            "to "
            f"{aggregate['aggregate_delta_pr_auc_ci_95']['high']:.6f}"
        ),
        f"Consistent seeds: {aggregate['consistent_seed_count']} / 3",
        f"AI VALUE GATE: {'PASS' if gate['passed'] else 'NOT PASSED'}",
    ]
)

save_text_evidence(
    "07_AI_VALUE_GATE.txt",
    ai_value_gate_text,
)

In [ ]:
# ============================================================
# 26. LOAD VALIDATION ALLOCATION REGRET
# ============================================================

with open(PATHS["baseline_allocation"], "r", encoding="utf-8") as handle:
    baseline_allocation = json.load(handle)

with open(PATHS["model_allocation"], "r", encoding="utf-8") as handle:
    model_allocation = json.load(handle)


def get_retained_ratio(item: dict) -> float:
    """
    Normalize evidence schema differences.

    Baseline evidence:
        oracle_value_retained

    Model evidence:
        economic_value_retained_ratio

    Older compatible evidence can also be reconstructed from totals.
    """
    if "oracle_value_retained" in item:
        return float(item["oracle_value_retained"])

    if "economic_value_retained_ratio" in item:
        return float(item["economic_value_retained_ratio"])

    oracle_total = item.get("oracle_value_total")
    system_total = item.get("system_oracle_value_total")

    if oracle_total is not None and system_total is not None:
        oracle_total = float(oracle_total)
        system_total = float(system_total)

        return (
            system_total / oracle_total
            if oracle_total > 0
            else 1.0
        )

    raise KeyError(
        "Allocation evidence tidak memiliki retained-value field yang dikenali."
    )


allocation_rows = []

for item in baseline_allocation["baselines"]:
    allocation_rows.append(
        {
            "System": item["baseline_id"],
            "Mean Regret": float(item["mean_regret"]),
            "Normalized Regret": float(item["mean_normalized_regret"]),
            "Exact Match Rate": float(item["exact_allocation_match_rate"]),
            "Oracle Value Retained": get_retained_ratio(item),
        }
    )

for item in model_allocation["models"]:
    raw_name = str(item["model"])

    display_name = {
        "LR": "LR",
        "HGB_B": "HGB-B",
        "HGB_E": "HGB-E",
    }.get(raw_name, raw_name)

    allocation_rows.append(
        {
            "System": display_name,
            "Mean Regret": float(item["mean_regret"]),
            "Normalized Regret": float(item["mean_normalized_regret"]),
            "Exact Match Rate": float(item["exact_allocation_match_rate"]),
            "Oracle Value Retained": get_retained_ratio(item),
        }
    )

allocation_df = (
    pd.DataFrame(allocation_rows)
    .drop_duplicates("System", keep="last")
    .set_index("System")
    .sort_index()
)

display(allocation_df)

save_text_evidence(
    "08_VALIDATION_ALLOCATION_REGRET.txt",
    allocation_df.to_string(),
)

In [ ]:
# ============================================================
# 27. VALIDATION MEAN ALLOCATION REGRET
# ============================================================

plt.figure(figsize=(9, 5))

allocation_df["Mean Regret"].sort_values().plot(
    kind="bar"
)

plt.title("Validation Mean Allocation Regret")
plt.xlabel("System")
plt.ylabel("Mean Regret (Lower is Better)")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("15_validation_allocation_regret.png")
plt.show()

In [ ]:
# ============================================================
# 28. VALIDATION ORACLE VALUE RETAINED
# ============================================================

plt.figure(figsize=(9, 5))

allocation_df["Oracle Value Retained"].sort_values().plot(
    kind="bar"
)

plt.title("Validation Oracle Value Retained")
plt.xlabel("System")
plt.ylabel("Oracle Value Retained")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("16_oracle_value_retained.png")
plt.show()

In [ ]:
# ============================================================
# 29. SELECTED MODEL MANIFEST
# ============================================================

with open(PATHS["selected_manifest"], "r", encoding="utf-8") as handle:
    selected_manifest = json.load(handle)

print("Selected label:", selected_manifest["selected_label"])
print("Model family:", selected_manifest["model_family"])
print("Model ID:", selected_manifest["model_id"])
print("Status:", selected_manifest["status"])
print("Training rows:", selected_manifest["training"]["rows"])
print("Training scenario groups:", selected_manifest["training"]["scenario_groups"])
print("Training duration (seconds):", round(selected_manifest["training"]["duration_seconds"], 6))

print("\nValidation reproduction:")
print("PR-AUC:", round(selected_manifest["selection_validation"]["pr_auc"], 6))
print("Brier:", round(selected_manifest["selection_validation"]["brier"], 6))
print("Artifact roundtrip verified:", selected_manifest["selection_validation"]["artifact_roundtrip_verified"])

print("\nArtifact SHA256:")
print(selected_manifest["artifact"]["sha256"])

selected_model_text = "\n".join(
    [
        f"Selected label: {selected_manifest['selected_label']}",
        f"Model family: {selected_manifest['model_family']}",
        f"Model ID: {selected_manifest['model_id']}",
        f"Status: {selected_manifest['status']}",
        f"Training rows: {selected_manifest['training']['rows']}",
        (
            "Training scenario groups: "
            f"{selected_manifest['training']['scenario_groups']}"
        ),
        (
            "Training duration (seconds): "
            f"{selected_manifest['training']['duration_seconds']:.6f}"
        ),
        "",
        "Validation reproduction:",
        (
            "PR-AUC: "
            f"{selected_manifest['selection_validation']['pr_auc']:.6f}"
        ),
        (
            "Brier: "
            f"{selected_manifest['selection_validation']['brier']:.6f}"
        ),
        (
            "Artifact roundtrip verified: "
            f"{selected_manifest['selection_validation']['artifact_roundtrip_verified']}"
        ),
        "",
        "Artifact SHA256:",
        selected_manifest["artifact"]["sha256"],
    ]
)

save_text_evidence(
    "09_SELECTED_MODEL_MANIFEST.txt",
    selected_model_text,
)

In [ ]:
# ============================================================
# 30. LOAD FINAL LOCKED TEST EVIDENCE
# ============================================================

with open(PATHS["final_test"], "r", encoding="utf-8") as handle:
    final_test = json.load(handle)

print("Benchmark run:", final_test["benchmark_run_id"])
print("Status:", final_test["status"])
print("Selected model:", final_test["selected_model"])
print("Selection frozen before test:", final_test["selection_frozen_before_test"])
print("Test accessed:", final_test["test_accessed"])
print("Test rows:", final_test["test_rows"])
print("Test scenario groups:", final_test["test_scenario_groups"])

In [ ]:
# ============================================================
# 31. FINAL TEST PREDICTIVE METRICS
# ============================================================

final_predictive_df = (
    pd.DataFrame(final_test["predictive_metrics"])
    .T
)

display(final_predictive_df)

In [ ]:
# ============================================================
# 32. FINAL TEST PR-AUC COMPARISON
# ============================================================

plt.figure(figsize=(8, 5))

final_predictive_df["pr_auc"].plot(
    kind="bar"
)

plt.title("Final Locked Test PR-AUC")
plt.xlabel("System")
plt.ylabel("PR-AUC")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("17_final_test_pr_auc.png")
plt.show()

In [ ]:
# ============================================================
# 33. FINAL TEST BRIER COMPARISON
# ============================================================

plt.figure(figsize=(8, 5))

final_predictive_df["brier"].plot(
    kind="bar"
)

plt.title("Final Locked Test Brier Score")
plt.xlabel("System")
plt.ylabel("Brier Score (Lower is Better)")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("18_final_test_brier.png")
plt.show()

In [ ]:
# ============================================================
# 34. FINAL TEST RANKING METRICS
# ============================================================

final_ranking_df = (
    pd.DataFrame(final_test["ranking_metrics"])
    .T
)

display(final_ranking_df)

final_ranking_df[
    [
        "mrr",
        "ndcg_at_3",
        "top1_success_rate",
    ]
].plot(
    kind="bar",
    figsize=(9, 5),
)

plt.title("Final Locked Test Ranking Quality")
plt.xlabel("System")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("19_final_test_ranking.png")
plt.show()

In [ ]:
# ============================================================
# 35. FINAL TEST ALLOCATION METRICS
# ============================================================

final_allocation_df = (
    pd.DataFrame(final_test["allocation_metrics"])
    .T
)

display(final_allocation_df)

final_locked_test_text = "\n".join(
    [
        f"Benchmark run: {final_test['benchmark_run_id']}",
        f"Status: {final_test['status']}",
        f"Selected model: {final_test['selected_model']}",
        (
            "Selection frozen before test: "
            f"{final_test['selection_frozen_before_test']}"
        ),
        f"Test accessed: {final_test['test_accessed']}",
        f"Test rows: {final_test['test_rows']}",
        f"Test scenario groups: {final_test['test_scenario_groups']}",
        "",
        "PREDICTIVE METRICS",
        "=" * 70,
        final_predictive_df.to_string(),
        "",
        "RANKING METRICS",
        "=" * 70,
        final_ranking_df.to_string(),
        "",
        "ALLOCATION METRICS",
        "=" * 70,
        final_allocation_df.to_string(),
        "",
        "SAFETY",
        "=" * 70,
        f"Quantity conservation: {final_test['safety']['quantity_conservation']}",
        (
            "Hard constraint violations: "
            f"{final_test['safety']['hard_constraint_violations']}"
        ),
    ]
)

save_text_evidence(
    "10_FINAL_LOCKED_TEST.txt",
    final_locked_test_text,
)

In [ ]:
# ============================================================
# 36. FINAL TEST MEAN ALLOCATION REGRET
# ============================================================

plt.figure(figsize=(8, 5))

final_allocation_df["mean_regret"].plot(
    kind="bar"
)

plt.title("Final Locked Test Mean Allocation Regret")
plt.xlabel("System")
plt.ylabel("Mean Regret (Lower is Better)")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("20_final_test_allocation_regret.png")
plt.show()

In [ ]:
# ============================================================
# 37. FINAL TEST ORACLE VALUE RETAINED
# ============================================================

plt.figure(figsize=(8, 5))

final_allocation_df["oracle_value_retained"].plot(
    kind="bar"
)

plt.title("Final Locked Test Oracle Value Retained")
plt.xlabel("System")
plt.ylabel("Oracle Value Retained")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("21_final_test_oracle_retained.png")
plt.show()

In [ ]:
# ============================================================
# 38. LOAD FINAL TEST PREDICTIONS
# ============================================================

final_predictions = pd.read_csv(
    PATHS["final_predictions"]
)

print("Final prediction rows:", len(final_predictions))
print("Scenario groups:", final_predictions["scenario_group_id"].nunique())

display(final_predictions.head())

In [ ]:
# ============================================================
# 39. FINAL TEST SCORE DISTRIBUTION
# ============================================================

plt.figure(figsize=(8, 5))

plt.hist(
    final_predictions["hgb_score"],
    bins=20,
    alpha=0.6,
    label="HGB-E",
)

plt.hist(
    final_predictions["b1_score"],
    bins=20,
    alpha=0.6,
    label="B1",
)

plt.title("Final Test Probability Score Distribution")
plt.xlabel("Predicted Rescue Probability")
plt.ylabel("Candidate Rows")
plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("22_score_distribution.png")
plt.show()

In [ ]:
# ============================================================
# 40. FINAL TEST RELIABILITY TABLE
# ============================================================

def calibration_table(
    frame: pd.DataFrame,
    score_column: str,
    bins: int = 10,
) -> pd.DataFrame:
    temp = frame[
        [
            score_column,
            "simulated_rescue_outcome",
        ]
    ].copy()

    temp["bin"] = pd.cut(
        temp[score_column],
        bins=np.linspace(0, 1, bins + 1),
        include_lowest=True,
    )

    return (
        temp.groupby("bin", observed=False)
        .agg(
            mean_predicted=(score_column, "mean"),
            observed_rate=("simulated_rescue_outcome", "mean"),
            rows=("simulated_rescue_outcome", "size"),
        )
        .dropna(
            subset=[
                "mean_predicted",
                "observed_rate",
            ]
        )
        .reset_index(drop=True)
    )


cal_hgb = calibration_table(
    final_predictions,
    "hgb_score",
)

cal_b1 = calibration_table(
    final_predictions,
    "b1_score",
)

print("HGB-E calibration table:")
display(cal_hgb)

print("B1 calibration table:")
display(cal_b1)

In [ ]:
# ============================================================
# 41. FINAL TEST RELIABILITY DIAGRAM
# ============================================================

plt.figure(figsize=(7, 7))

plt.plot(
    cal_hgb["mean_predicted"],
    cal_hgb["observed_rate"],
    marker="o",
    label="HGB-E",
)

plt.plot(
    cal_b1["mean_predicted"],
    cal_b1["observed_rate"],
    marker="o",
    label="B1",
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1,
    label="Ideal Calibration",
)

plt.title("Final Locked Test Reliability Diagram")
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Observed Rescue Outcome Rate")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
save_current_figure("23_reliability_diagram.png")
plt.show()

In [ ]:
# ============================================================
# 42. FINAL RESULT SUMMARY
# ============================================================

hgb_pr_auc = final_predictive_df.loc["HGB_E", "pr_auc"]
b1_pr_auc = final_predictive_df.loc["B1", "pr_auc"]

hgb_brier = final_predictive_df.loc["HGB_E", "brier"]
b1_brier = final_predictive_df.loc["B1", "brier"]

hgb_regret = final_allocation_df.loc["HGB_E", "mean_regret"]
b1_regret = final_allocation_df.loc["B1", "mean_regret"]

summary_df = pd.DataFrame(
    {
        "Metric": [
            "PR-AUC",
            "Brier",
            "ROC-AUC",
            "MRR",
            "NDCG@3",
            "Top-1 Success Rate",
            "Mean Allocation Regret",
            "Oracle Value Retained",
        ],
        "HGB-E": [
            final_predictive_df.loc["HGB_E", "pr_auc"],
            final_predictive_df.loc["HGB_E", "brier"],
            final_predictive_df.loc["HGB_E", "roc_auc"],
            final_ranking_df.loc["HGB_E", "mrr"],
            final_ranking_df.loc["HGB_E", "ndcg_at_3"],
            final_ranking_df.loc["HGB_E", "top1_success_rate"],
            final_allocation_df.loc["HGB_E", "mean_regret"],
            final_allocation_df.loc["HGB_E", "oracle_value_retained"],
        ],
        "B1": [
            final_predictive_df.loc["B1", "pr_auc"],
            final_predictive_df.loc["B1", "brier"],
            final_predictive_df.loc["B1", "roc_auc"],
            final_ranking_df.loc["B1", "mrr"],
            final_ranking_df.loc["B1", "ndcg_at_3"],
            final_ranking_df.loc["B1", "top1_success_rate"],
            final_allocation_df.loc["B1", "mean_regret"],
            final_allocation_df.loc["B1", "oracle_value_retained"],
        ],
    }
)

display(summary_df)

print("\nSelected model: HGB-E")
print("AI Value Gate: PASS")
print("Final PR-AUC improvement:", round(hgb_pr_auc - b1_pr_auc, 6))
print(
    "Final Brier difference:",
    round(hgb_brier - b1_brier, 6),
    "(negative = better)",
)
print(
    "Allocation regret reduction:",
    round((1 - hgb_regret / b1_regret) * 100, 2),
    "%",
)

print("\nQuantity conservation:", final_test["safety"]["quantity_conservation"])
print("Hard constraint violations:", final_test["safety"]["hard_constraint_violations"])

final_summary_text = "\n".join(
    [
        summary_df.to_string(index=False),
        "",
        "Selected model: HGB-E",
        "AI Value Gate: PASS",
        f"Final PR-AUC improvement: {hgb_pr_auc - b1_pr_auc:+.6f}",
        (
            "Final Brier difference: "
            f"{hgb_brier - b1_brier:+.6f} "
            "(negative = better)"
        ),
        (
            "Allocation regret reduction: "
            f"{(1 - hgb_regret / b1_regret) * 100:.2f}%"
        ),
        "",
        (
            "Quantity conservation: "
            f"{final_test['safety']['quantity_conservation']}"
        ),
        (
            "Hard constraint violations: "
            f"{final_test['safety']['hard_constraint_violations']}"
        ),
    ]
)

save_text_evidence(
    "11_FINAL_SUMMARY.txt",
    final_summary_text,
)

In [ ]:
# ============================================================
# 43. CLAIM BOUNDARY
# ============================================================

print(
    "SUPPORTED CLAIM:\n"
    "HGB-E is the selected feature-aware rescue-success model for Afterlife AI. "
    "It passed the AI Value Gate on the frozen synthetic development benchmark "
    "and retained an advantage over B1 on the final locked synthetic test for "
    "predictive quality, ranking quality, and allocation regret, while quantity "
    "conservation remained PASS and hard-constraint violations remained zero."
)

print(
    "\nNOT SUPPORTED BY THIS NOTEBOOK:\n"
    "- real-world rescue probability accuracy\n"
    "- real-world transaction performance\n"
    "- real-world economic impact\n"
    "- real-world prevalence of operational constraints"
)

print(
    "\nIMPORTANT:\n"
    "This notebook is a reporting and visualization layer only. "
    "It must not be used to retrain, tune, recalibrate, or reselect the model "
    "after the final locked test has been consumed."
)

print(
    "\nPORTABILITY NOTE:\n"
    "Notebook uses repository-relative evidence paths. "
    "No developer-specific Windows path is required."
)

claim_boundary_text = (
    "SUPPORTED CLAIM:\n"
    "HGB-E is the selected feature-aware rescue-success model for Afterlife AI. "
    "It passed the AI Value Gate on the frozen synthetic development benchmark "
    "and retained an advantage over B1 on the final locked synthetic test for "
    "predictive quality, ranking quality, and allocation regret, while quantity "
    "conservation remained PASS and hard-constraint violations remained zero.\n\n"
    "NOT SUPPORTED BY THIS NOTEBOOK:\n"
    "- real-world rescue probability accuracy\n"
    "- real-world transaction performance\n"
    "- real-world economic impact\n"
    "- real-world prevalence of operational constraints\n\n"
    "This notebook is a reporting and visualization layer only. "
    "It must not be used to retrain, tune, recalibrate, or reselect the model "
    "after the final locked test has been consumed."
)

save_text_evidence(
    "12_CLAIM_BOUNDARY.txt",
    claim_boundary_text,
)

print("\nExport complete.")
print("Visualizations:", REPORT_DIR)
print("Text evidence:", TEXT_DIR)